# Actividad 5 - Clasificación: Titanic
Este notebook reproduce la preparación de datos, el entrenamiento de dos algoritmos (LogisticRegression y RandomForest), ajuste por GridSearch y registro en MLflow.


In [ ]:
# Instalar dependencias (en Colab)
!pip install -q scikit-learn pandas mlflow joblib

In [ ]:
# Descargar dataset (si no existe)
from pathlib import Path
p = Path('Activity5/datos/datos_ini/titanic.csv')
if not p.exists():
    Path('Activity5/datos/datos_ini').mkdir(parents=True, exist_ok=True)
    import requests
    url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
    r = requests.get(url)
    p.write_bytes(r.content)
    print('Downloaded')
else:
    print('Already downloaded')


In [ ]:
# Preparar datos usando la librería provista
from Activity5.fuentes.datos_prep import prepare_for_model
raw = 'Activity5/datos/datos_ini/titanic.csv'
clean = 'Activity5/datos/datos_limp/titanic_clean.csv'
df = prepare_for_model(raw, clean)
df.head()


In [ ]:
# Entrenamiento y registro en MLflow desde el notebook
import mlflow
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
import pandas as pd
import joblib

df = pd.read_csv(clean)
X = df.drop(columns=['Survived'])
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

mlflow.set_tracking_uri('file:./Activity5/mlruns')

def eval_model(est):
    y_pred = est.predict(X_test)
    y_proba = est.predict_proba(X_test)[:,1]
    return {
        'accuracy': accuracy_score(y_test,y_pred),
        'roc_auc': roc_auc_score(y_test,y_proba),
        'precision': precision_score(y_test,y_pred),
        'recall': recall_score(y_test,y_pred),
        'f1': f1_score(y_test,y_pred)
    }

# Logistic Regression
pipe = Pipeline([('scaler', StandardScaler()),('clf', LogisticRegression(max_iter=1000))])
param_grid = {'clf__C':[0.01,0.1,1,10]}
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
gs = GridSearchCV(pipe,param_grid,cv=cv,scoring='roc_auc')

with mlflow.start_run(run_name='logistic_regression'):
    mlflow.log_param('algo','LogisticRegression')
    gs.fit(X_train,y_train)
    mlflow.log_params(gs.best_params_)
    metrics = eval_model(gs.best_estimator_)
    for k,v in metrics.items():
        mlflow.log_metric(k,v)
    mlflow.sklearn.log_model(gs.best_estimator_,'model')
    print('LR metrics',metrics)

# Random Forest
pipe = Pipeline([('scaler', StandardScaler()),('clf', RandomForestClassifier(random_state=42))])
param_grid = {'clf__n_estimators':[50,100], 'clf__max_depth':[None,5,10]}
gs = GridSearchCV(pipe,param_grid,cv=cv,scoring='roc_auc',n_jobs=-1)
with mlflow.start_run(run_name='random_forest'):
    mlflow.log_param('algo','RandomForest')
    gs.fit(X_train,y_train)
    mlflow.log_params(gs.best_params_)
    metrics = eval_model(gs.best_estimator_)
    for k,v in metrics.items():
        mlflow.log_metric(k,v)
    mlflow.sklearn.log_model(gs.best_estimator_,'model')
    print('RF metrics',metrics)


## Resultados y análisis
Revisa la carpeta Activity5/mlruns generada por MLflow para ver los experimentos, parámetros, métricas y artefactos.